Author: Konstantin Malanchev and the LINCC Frameworks team

Last updated: November 2026

# Homework 2b: photo-z agreement sky map

**Task:** plot the fraction of objects with matching Legacy Survey and DP2 photo-z as a sky map.

**Plan:**
1. Open the catalog saved in `Notebook_2a_Crossmatch`
2. Compute the fraction of matching photo-z in each HEALPix pixel
3. Plot it as a sky map

If you'd like to solve it yourself, `Notebook_2a_Crossmatch` may help.

> [!NOTE]
> This notebook uses only the shareable catalog saved in `Notebook_2a_Crossmatch`, so anyone who has received this catalog can run it, including non-DRHs.

## Solution

In [ ]:
# lsdb is pre-installed on RSP, uncomment the line below if you are on another platform
# %pip install -qq lsdb

In [ ]:
# Standard library imports
import logging
import os
import warnings

# Third-party imports
import astropy.units as u
import lsdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from dask.distributed import Client
from hats.inspection.visualize_catalog import plot_healpix_map
from hats.pixel_math.spatial_index import spatial_index_to_healpix
from matplotlib.colors import Normalize

# Configuration
warnings.simplefilter("ignore")
logging.getLogger("distributed").setLevel(logging.WARNING)

## Setup Dask cluster

We use the same Dask setup as in `Notebook_1a_Intro`: sixteen workers, one per CPU of a "Large" RSP container, with one thread and 2 GB of memory each, which is enough for this low-memory pipeline, and `/deleted-sundays/` for spilling data to disk.

If you are not on RSP, set `local_directory` to a scratch directory on your platform.

In [ ]:
client = Client(
    n_workers=16,
    threads_per_worker=1,
    local_directory=f"/deleted-sundays/{os.environ.get('USER', 'dask_scratch')}",
)

# Print the dashboard link and port
print(f"Dask is running at: {client.dashboard_link}")

### Open the catalog

All of its columns are shareable, so we can look at the data.

In [ ]:
shared = lsdb.open_catalog("lsdr11_rubindp2_photoz_match")
shared.head()

### Fraction of matching photo-z per HEALPix pixel

The catalog index, `_healpix_29`, is the HEALPix pixel index at order 29. We convert it to a lower order and count objects in each pixel with `map_partitions()`, as we did for the histograms in `Notebook_1a_Intro`. A pixel may span several partitions, so we sum the partial counts at the end. Order 9 corresponds to pixels of about 6.9 arcmin. We skip pixels with too few objects.

In [ ]:
ORDER = 9
MIN_COUNT = 10


def per_pixel_counts(df):
    """Number of objects and number of matching photo-z per HEALPix pixel for a single partition."""
    pixel = spatial_index_to_healpix(df.index.to_numpy(), target_order=ORDER)
    counts = df["lsdr11_rubindp2_photoz_match"].groupby(pixel).agg(["size", "sum"])
    return pd.DataFrame(
        {"pixel": counts.index, "n": counts["size"].to_numpy(), "n_match": counts["sum"].to_numpy()}
    )


partial_counts = shared.map_partitions(per_pixel_counts, meta={"pixel": int, "n": int, "n_match": int})

In [ ]:
per_pixel = partial_counts.compute().groupby("pixel")[["n", "n_match"]].sum()
per_pixel = per_pixel[per_pixel["n"] >= MIN_COUNT]
per_pixel["fraction"] = per_pixel["n_match"] / per_pixel["n"]
print(f"Number of pixels: {len(per_pixel):,}")

### Sky map

We plot the fractions with `plot_healpix_map()`, see the ["Plotting aggregations of data"](https://docs.lsdb.io/en/latest/tutorials/pre_executed/plotting.html#5.-Plotting-aggregations-of-data) section of the LSDB plotting tutorial for more examples.

In [ ]:
fig, ax = plot_healpix_map(
    per_pixel["fraction"].to_numpy(),
    ipix=per_pixel.index.to_numpy(),
    depth=np.full(len(per_pixel), ORDER),
    fov=(12 * u.deg, 12 * u.deg),
    center=SkyCoord(300.0, -20.0, unit="deg"),
    norm=Normalize(vmin=0.0, vmax=1.0),
    title="Fraction of matching Legacy Survey and DP2 photo-z",
)
plt.show()

In [ ]:
client.close()

## Citing

If you use the tools or the data from this notebook in your research, please cite the following papers.

<details>
<summary>For LSDB and HATS cite: Caplar et al. (2025), arXiv:2501.02103</summary>

```bibtex
% ADS bibcode: 2025arXiv250102103C
@ARTICLE{Caplar_LSDB_2025,
       author = {{Caplar}, Neven and {Beebe}, Wilson and {Branton}, Doug and {Campos}, Sandro and {Connolly}, Andrew and {DeLucchi}, Melissa and {Jones}, Derek and {Juric}, Mario and {Kubica}, Jeremy and {Malanchev}, Konstantin and {Mandelbaum}, Rachel and {McGuire}, Sean},
        title = "{Using LSDB to enable large-scale catalog distribution, cross-matching, and analytics}",
      journal = {arXiv e-prints},
     keywords = {Instrumentation and Methods for Astrophysics},
         year = 2025,
        month = jan,
          eid = {arXiv:2501.02103},
        pages = {arXiv:2501.02103},
          doi = {10.48550/arXiv.2501.02103},
archivePrefix = {arXiv},
       eprint = {2501.02103},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2025arXiv250102103C},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For Rubin catalogs in HATS cite: Malanchev et al. (2025), arXiv:2506.23955</summary>

```bibtex
% ADS bibcode: 2025arXiv250623955M
@ARTICLE{Malanchev_DP1_2025,
       author = {{Malanchev}, Konstantin and {DeLucchi}, Melissa and {Caplar}, Neven and {Malz}, Alex I. and {Alexov}, Anastasia and {Aubourg}, Eric and {E Bauer}, Amanda and {Beebe}, Wilson and {Bellm}, Eric C. and {Blum}, Robert David and {Branton}, Doug and {Campos}, Sandro and {Calabrese}, Daniel and {Carlin}, Jeffrey L. and {Choi}, Yumi and {Connolly}, Andrew and {Dai}, Mi and {Daly}, Philip N. and {Daruich}, Felipe and {Daubard}, Guillaume and {Delgado}, Francisco and {Drass}, Holger and {Fonseca Alvarez}, Gloria and {Gangler}, Emmanuel and {Guy}, Leanne P. and {Ivezi{\'c}}, {\v{Z}}eljko and {Jammes}, Fabrice and {Jannuzi}, Buell T. and {Jenness}, Tim and {Jimenez}, David and {Jones}, Derek T. and {Juri{\'c}}, Mario and {Kahn}, Steven M. and {Kang}, Yijung and {Kannawadi}, Arun and {Kubica}, Jeremy and {Le Guillou}, Laurent and {Lynn}, Olivia and {Mandelbaum}, Rachel and {Moniez}, Marc and {Morrison}, Christopher B. and {McGuire}, Sean and {Ordenes}, Ian E. and {Patterson}, Maria T. and {Plazas Malag{\'o}n}, Andr{\'e}s A. and {Qiu}, Yongqiang and {Quint}, Bruno and {Radeka}, Veljko and {van Reeven}, Wouter and {Riot}, Vincent J. and {S{\'a}nchez}, Bruno O. and {Sanmartim}, David and {Sebag}, Jacques and {Shugart}, Alysha and {Sullivan}, Ian S. and {Taranu}, Dan S. and {Urbach}, Elana K. and {Wood-Vasey}, W.~M. and {Zhang}, Tianqing},
        title = "{Variability-finding in Rubin Data Preview 1 with LSDB}",
      journal = {arXiv e-prints},
     keywords = {Instrumentation and Methods for Astrophysics, Solar and Stellar Astrophysics},
         year = 2025,
        month = jun,
          eid = {arXiv:2506.23955},
        pages = {arXiv:2506.23955},
          doi = {10.48550/arXiv.2506.23955},
archivePrefix = {arXiv},
       eprint = {2506.23955},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2025arXiv250623955M},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For Rubin DP2 data cite: Rubin Observatory (2026), RTN-115, doi:10.71929/rubin/3377440; DP2 dataset, doi:10.71929/rubin/3382528</summary>

```bibtex
% ADS bibcode: 2026rubn.rept....2V
@TECHREPORT{Rubin_DP2_2026,
       author = {{Vera C. Rubin Observatory Team}},
        title = "{RTN-115: The Vera C. Rubin Observatory Data Preview 2}",
  institution = {NSF-DOE Vera C. Rubin Observatory},
     keywords = {79 ASTRONOMY AND ASTROPHYSICS},
         year = 2026,
 howpublished = {NSF-DOE Vera C. Rubin Observatory Technical Report},
          doi = {10.71929/RUBIN/3377440},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026rubn.rept....2V},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2026lsst.data...42N
@dataset{Rubin_DP2Data_2026,
       author = {{NSF-DOE Vera C. Rubin Observatory}},
        title = "{Legacy Survey of Space and Time Data Preview 2}",
 howpublished = {NSF-DOE Vera C. Rubin Observatory Data Set},
         year = 2026,
        month = jan,
          doi = {10.71929/RUBIN/3382528},
    publisher = {Rubin Obs},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026lsst.data...42N},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For DESI Legacy Survey DR11 cite: Dey et al. (2019), AJ, 157, 168</summary>

```bibtex
% ADS bibcode: 2019AJ....157..168D
@ARTICLE{Dey_LegacySurveys_2019,
       author = {{Dey}, Arjun and {Schlegel}, David J. and {Lang}, Dustin and {Blum}, Robert and {Burleigh}, Kaylan and {Fan}, Xiaohui and {Findlay}, Joseph R. and {Finkbeiner}, Doug and {Herrera}, David and {Juneau}, St{\'e}phanie and {Landriau}, Martin and {Levi}, Michael and {McGreer}, Ian and {Meisner}, Aaron and {Myers}, Adam D. and {Moustakas}, John and {Nugent}, Peter and {Patej}, Anna and {Schlafly}, Edward F. and {Walker}, Alistair R. and {Valdes}, Francisco and {Weaver}, Benjamin A. and {Y{\`e}che}, Christophe and {Zou}, Hu and {Zhou}, Xu and {Abareshi}, Behzad and {Abbott}, T.~M.~C. and {Abolfathi}, Bela and {Aguilera}, C. and {Alam}, Shadab and {Allen}, Lori and {Alvarez}, A. and {Annis}, James and {Ansarinejad}, Behzad and {Aubert}, Marie and {Beechert}, Jacqueline and {Bell}, Eric F. and {BenZvi}, Segev Y. and {Beutler}, Florian and {Bielby}, Richard M. and {Bolton}, Adam S. and {Brice{\~n}o}, C{\'e}sar and {Buckley-Geer}, Elizabeth J. and {Butler}, Karen and {Calamida}, Annalisa and {Carlberg}, Raymond G. and {Carter}, Paul and {Casas}, Ricard and {Castander}, Francisco J. and {Choi}, Yumi and {Comparat}, Johan and {Cukanovaite}, Elena and {Delubac}, Timoth{\'e}e and {DeVries}, Kaitlin and {Dey}, Sharmila and {Dhungana}, Govinda and {Dickinson}, Mark and {Ding}, Zhejie and {Donaldson}, John B. and {Duan}, Yutong and {Duckworth}, Christopher J. and {Eftekharzadeh}, Sarah and {Eisenstein}, Daniel J. and {Etourneau}, Thomas and {Fagrelius}, Parker A. and {Farihi}, Jay and {Fitzpatrick}, Mike and {Font-Ribera}, Andreu and {Fulmer}, Leah and {G{\"a}nsicke}, Boris T. and {Gaztanaga}, Enrique and {George}, Koshy and {Gerdes}, David W. and {Gontcho}, Satya Gontcho A. and {Gorgoni}, Claudio and {Green}, Gregory and {Guy}, Julien and {Harmer}, Diane and {Hernandez}, M. and {Honscheid}, Klaus and {Huang}, Lijuan Wendy and {James}, David J. and {Jannuzi}, Buell T. and {Jiang}, Linhua and {Joyce}, Richard and {Karcher}, Armin and {Karkar}, Sonia and {Kehoe}, Robert and {Kneib}, Jean-Paul and {Kueter-Young}, Andrea and {Lan}, Ting-Wen and {Lauer}, Tod R. and {Le Guillou}, Laurent and {Le Van Suu}, Auguste and {Lee}, Jae Hyeon and {Lesser}, Michael and {Perreault Levasseur}, Laurence and {Li}, Ting S. and {Mann}, Justin L. and {Marshall}, Robert and {Mart{\'\i}nez-V{\'a}zquez}, C.~E. and {Martini}, Paul and {du Mas des Bourboux}, H{\'e}lion and {McManus}, Sean and {Meier}, Tobias Gabriel and {M{\'e}nard}, Brice and {Metcalfe}, Nigel and {Mu{\~n}oz-Guti{\'e}rrez}, Andrea and {Najita}, Joan and {Napier}, Kevin and {Narayan}, Gautham and {Newman}, Jeffrey A. and {Nie}, Jundan and {Nord}, Brian and {Norman}, Dara J. and {Olsen}, Knut A.~G. and {Paat}, Anthony and {Palanque-Delabrouille}, Nathalie and {Peng}, Xiyan and {Poppett}, Claire L. and {Poremba}, Megan R. and {Prakash}, Abhishek and {Rabinowitz}, David and {Raichoor}, Anand and {Rezaie}, Mehdi and {Robertson}, A.~N. and {Roe}, Natalie A. and {Ross}, Ashley J. and {Ross}, Nicholas P. and {Rudnick}, Gregory and {Safonova}, Sasha and {Saha}, Abhijit and {S{\'a}nchez}, F. Javier and {Savary}, Elodie and {Schweiker}, Heidi and {Scott}, Adam and {Seo}, Hee-Jong and {Shan}, Huanyuan and {Silva}, David R. and {Slepian}, Zachary and {Soto}, Christian and {Sprayberry}, David and {Staten}, Ryan and {Stillman}, Coley M. and {Stupak}, Robert J. and {Summers}, David L. and {Sien Tie}, Suk and {Tirado}, H. and {Vargas-Maga{\~n}a}, Mariana and {Vivas}, A. Katherina and {Wechsler}, Risa H. and {Williams}, Doug and {Yang}, Jinyi and {Yang}, Qian and {Yapici}, Tolga and {Zaritsky}, Dennis and {Zenteno}, A. and {Zhang}, Kai and {Zhang}, Tianmeng and {Zhou}, Rongpu and {Zhou}, Zhimin},
        title = "{Overview of the DESI Legacy Imaging Surveys}",
      journal = {\aj},
     keywords = {catalogs, surveys, Astrophysics - Instrumentation and Methods for Astrophysics},
         year = 2019,
        month = may,
       volume = {157},
       number = {5},
          eid = {168},
        pages = {168},
          doi = {10.3847/1538-3881/ab089d},
archivePrefix = {arXiv},
       eprint = {1804.08657},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2019AJ....157..168D},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For DESI Legacy Survey photo-z cite: Zhou et al. (2023), AJ, 165, 58</summary>

```bibtex
% ADS bibcode: 2023AJ....165...58Z
@ARTICLE{Zhou_DESI_2023,
       author = {{Zhou}, Rongpu and {Dey}, Biprateep and {Newman}, Jeffrey A. and {Eisenstein}, Daniel J. and {Dawson}, K. and {Bailey}, S. and {Berti}, A. and {Guy}, J. and {Lan}, Ting-Wen and {Zou}, H. and {Aguilar}, J. and {Ahlen}, S. and {Alam}, Shadab and {Brooks}, D. and {de la Macorra}, A. and {Dey}, A. and {Dhungana}, G. and {Fanning}, K. and {Font-Ribera}, A. and {Gontcho}, S. Gontcho A. and {Honscheid}, K. and {Ishak}, Mustapha and {Kisner}, T. and {Kov{\'a}cs}, A. and {Kremin}, A. and {Landriau}, M. and {Levi}, Michael E. and {Magneville}, C. and {Manera}, Marc and {Martini}, P. and {Meisner}, Aaron M. and {Miquel}, R. and {Moustakas}, J. and {Myers}, Adam D. and {Nie}, Jundan and {Palanque-Delabrouille}, N. and {Percival}, W.~J. and {Poppett}, C. and {Prada}, F. and {Raichoor}, A. and {Ross}, A.~J. and {Schlafly}, E. and {Schlegel}, D. and {Schubnell}, M. and {Tarl{\'e}}, Gregory and {Weaver}, B.~A. and {Wechsler}, R.~H. and {Y{\'e}che}, Christophe and {Zhou}, Zhimin},
        title = "{Target Selection and Validation of DESI Luminous Red Galaxies}",
      journal = {\aj},
     keywords = {Cosmology, Redshift surveys, Astrophysics - Cosmology and Nongalactic Astrophysics, Astrophysics - Astrophysics of Galaxies},
         year = 2023,
        month = feb,
       volume = {165},
       number = {2},
          eid = {58},
        pages = {58},
          doi = {10.3847/1538-3881/aca5fb},
archivePrefix = {arXiv},
       eprint = {2208.08515},
 primaryClass = {astro-ph.CO},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2023AJ....165...58Z},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For Rubin DP2 photo-z cite: Charles et al. (2026), RTN-124, doi:10.71929/rubin/3484518; RAIL Team et al. (2026), OJAp, 9, 58200; Dalmasso et al. (2020), A&C, 30, 100362</summary>

```bibtex
% ADS bibcode: 2026rubn.rept...11C
@TECHREPORT{Charles_RTN124_2026,
       author = {{Charles}, Eric and {Zhang}, Tianqing and {Schmidt}, Samuel J. and {Taranu}, Dan S. and {Gschwend}, Julia and {Graham}, Melissa L. and {Kannawadi}, Arun},
        title = "{RTN-124: Photometric Redshifts for the Vera C. Rubin Observatory Data Preview}",
  institution = {NSF-DOE Vera C. Rubin Observatory},
     keywords = {79 ASTRONOMY AND ASTROPHYSICS},
         year = 2026,
 howpublished = {NSF-DOE Vera C. Rubin Observatory Technical Report},
          doi = {10.71929/RUBIN/3484518},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026rubn.rept...11C},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2026OJAp....958200R
@ARTICLE{RAIL_Photoz_2026,
       author = {{RAIL Team} and {van den Busch}, Jan Luca and {Charles}, Eric and {Cohen-Tanugi}, Johann and {Crafford}, Alice and {Crenshaw}, John Franklin and {Dagoret}, Sylvie and {De-Santiago}, Josue and {De Vicente}, Juan and {Hang}, Qianjun and {Joachimi}, Benjamin and {Joudaki}, Shahab and {Kalmbach}, J. Bryce and {Kannawadi}, Arun and {Liang}, Shuang and {Lynn}, Olivia and {Malz}, Alex I. and {Mandelbaum}, Rachel and {Mau}, Sidney and {Medina-Varela}, Leonel and {Merz}, Grant and {Moskowitz}, Irene and {Oldag}, Drew and {Ruiz-Zapatero}, Jaime and {Rahman}, Mubdi and {Rau}, Markus M. and {Schmidt}, Samuel J. and {Scora}, Jennifer and {Shirley}, Raphael and {St{\"o}lzner}, Benjamin and {Cipriano}, Laura Toribio San and {Tortorelli}, Luca and {Yan}, Ziang and {Zhang}, Tianqing and {LSST Dark Energy Science Collaboration}},
        title = "{Redshift Assessment Infrastructure Layers (RAIL): Rubin-era photometric redshift stress-testing and at-scale production}",
      journal = {The Open Journal of Astrophysics},
     keywords = {Instrumentation and Methods for Astrophysics, Cosmology and Nongalactic Astrophysics, Astrophysics of Galaxies},
         year = 2026,
        month = feb,
       volume = {9},
        pages = {58200},
          doi = {10.33232/001c.158200},
archivePrefix = {arXiv},
       eprint = {2505.02928},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026OJAp....958200R},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2020A&C....3000362D
@ARTICLE{Dalmasso_FlexZBoost_2020,
       author = {{Dalmasso}, N. and {Pospisil}, T. and {Lee}, A.~B. and {Izbicki}, R. and {Freeman}, P.~E. and {Malz}, A.~I.},
        title = "{Conditional density estimation tools in python and R with applications to photometric redshifts and likelihood-free cosmological inference}",
      journal = {Astronomy and Computing},
     keywords = {Nonparametric statistics, Statistical software, Statistical computing, Methods: Data analysis, Galaxies: Distances and redshifts, Cosmology: Cosmological parameters, Astrophysics - Instrumentation and Methods for Astrophysics, Statistics - Computation, Statistics - Machine Learning},
         year = 2020,
        month = jan,
       volume = {30},
          eid = {100362},
        pages = {100362},
          doi = {10.1016/j.ascom.2019.100362},
archivePrefix = {arXiv},
       eprint = {1908.11523},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2020A&C....3000362D},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>